In [2]:
import sqlite3

# -----------------------------
# 1. Connect to SQLite database
# -----------------------------
conn = sqlite3.connect("star_schema.db")
cursor = conn.cursor()

# -----------------------------
# 2. Create Dimension Tables
# -----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY,
    date TEXT,
    year INTEGER,
    month TEXT,
    day INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_product (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    region TEXT
)
""")

# -----------------------------
# 3. Create Fact Table
# -----------------------------
cursor.execute("""
CREATE TABLE IF NOT EXISTS fact_sales (
    sales_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_id INTEGER,
    product_id INTEGER,
    customer_id INTEGER,
    quantity INTEGER,
    total_amount REAL,
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id),
    FOREIGN KEY (product_id) REFERENCES dim_product(product_id),
    FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id)
)
""")

# -----------------------------
# 4. Insert Demo Data
# -----------------------------
cursor.executemany("""
INSERT OR IGNORE INTO dim_date VALUES (?, ?, ?, ?, ?)
""", [
    (1, '2025-01-01', 2025, 'January', 1),
    (2, '2025-01-02', 2025, 'January', 2),
    (3, '2025-01-03', 2025, 'January', 3)
])

cursor.executemany("""
INSERT OR IGNORE INTO dim_product VALUES (?, ?, ?)
""", [
    (101, 'Laptop', 'Electronics'),
    (102, 'Mobile', 'Electronics'),
    (103, 'Chair', 'Furniture')
])

cursor.executemany("""
INSERT OR IGNORE INTO dim_customer VALUES (?, ?, ?)
""", [
    (201, 'Alice', 'East'),
    (202, 'Bob', 'West'),
    (203, 'Charlie', 'North')
])

cursor.executemany("""
INSERT INTO fact_sales (date_id, product_id, customer_id, quantity, total_amount)
VALUES (?, ?, ?, ?, ?)
""", [
    (1, 101, 201, 1, 80000),
    (1, 102, 202, 2, 40000),
    (2, 103, 203, 5, 15000),
    (3, 101, 202, 1, 78000),
    (3, 102, 201, 3, 60000)
])

conn.commit()

# -----------------------------
# 5. Print Dimension Tables
# -----------------------------
def print_table(table_name):
    print(f"\n📋 {table_name}")
    print("-" * 40)
    for row in cursor.execute(f"SELECT * FROM {table_name}"):
        print(row)

print_table("dim_date")
print_table("dim_product")
print_table("dim_customer")

# -----------------------------
# 6. Print Fact Table
# -----------------------------
print_table("fact_sales")

# -----------------------------
# 7. Analytical Queries
# -----------------------------

print("\n📊 Total Sales Amount by Product")
query1 = """
SELECT p.product_name, SUM(f.total_amount) AS total_sales
FROM fact_sales f
JOIN dim_product p ON f.product_id = p.product_id
GROUP BY p.product_name
"""
for row in cursor.execute(query1):
    print(row)

print("\n📊 Total Sales by Region")
query2 = """
SELECT c.region, SUM(f.total_amount) AS total_sales
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.region
"""
for row in cursor.execute(query2):
    print(row)

print("\n📊 Monthly Sales Summary")
query3 = """
SELECT d.month, SUM(f.total_amount) AS total_sales
FROM fact_sales f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.month
"""
for row in cursor.execute(query3):
    print(row)

# -----------------------------
# 8. Close Connection
# -----------------------------
conn.close()


📋 dim_date
----------------------------------------
(1, '2025-01-01', 2025, 'January', 1)
(2, '2025-01-02', 2025, 'January', 2)
(3, '2025-01-03', 2025, 'January', 3)

📋 dim_product
----------------------------------------
(101, 'Laptop', 'Electronics')
(102, 'Mobile', 'Electronics')
(103, 'Chair', 'Furniture')

📋 dim_customer
----------------------------------------
(201, 'Alice', 'East')
(202, 'Bob', 'West')
(203, 'Charlie', 'North')

📋 fact_sales
----------------------------------------
(1, 1, 101, 201, 1, 80000.0)
(2, 1, 102, 202, 2, 40000.0)
(3, 2, 103, 203, 5, 15000.0)
(4, 3, 101, 202, 1, 78000.0)
(5, 3, 102, 201, 3, 60000.0)
(6, 1, 101, 201, 1, 80000.0)
(7, 1, 102, 202, 2, 40000.0)
(8, 2, 103, 203, 5, 15000.0)
(9, 3, 101, 202, 1, 78000.0)
(10, 3, 102, 201, 3, 60000.0)

📊 Total Sales Amount by Product
('Chair', 30000.0)
('Laptop', 316000.0)
('Mobile', 200000.0)

📊 Total Sales by Region
('East', 280000.0)
('North', 30000.0)
('West', 236000.0)

📊 Monthly Sales Summary
('January', 5